In [11]:
!python --version

Python 3.13.11


In [12]:
import pandas as pd
from sqlalchemy import create_engine, text

In [13]:
schema = "liane_library"
host = "127.0.0.1"
user = "root"
password = "Satish425524"
port = 3306

connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"

engine = create_engine(connection_string)

In [14]:

books_df = pd.read_sql("""
    SELECT *
    FROM books;
""", con=engine)

friends_df = pd.read_sql("""
    SELECT *
    FROM friends;
""", con=engine)

book_loans_df = pd.read_sql("""
    SELECT *
    FROM book_loans;
""", con=engine)


In [15]:
books_df = pd.read_sql("SELECT * FROM books;", con=engine)
friends_df = pd.read_sql("SELECT * FROM friends;", con=engine)
loans_df = pd.read_sql("SELECT * FROM book_loans;", con=engine)

In [16]:
books_df.to_csv("books_export.csv", index=False)
friends_df.to_csv("friends_export.csv", index=False)
loans_df.to_csv("book_loans_export.csv", index=False)

In [17]:
borrowed_books_df = pd.read_sql("""
    SELECT 
        bl.loan_id,
        b.book_title,
        f.first_name,
        f.last_name,
        bl.loan_date,
        bl.due_date,
        bl.return_date
    FROM book_loans AS bl
    JOIN books AS b
        ON bl.book_id = b.book_id
    JOIN friends AS f
        ON bl.friend_id = f.friend_id
    WHERE bl.return_date IS NULL;
""", con=engine)

borrowed_books_df

,loan_id,book_title,first_name,last_name,loan_date,due_date,return_date
0,29,Harry Potter and the Philosopher's Stone,Liane,Müller,2026-06-01,2026-06-15,None
1,30,Beloved,Michael,Weber,2026-06-01,2026-06-15,None
2,31,The Shining,Anna,Fischer,2026-06-01,2026-06-15,None
3,32,Norwegian Wood,Liane,Müller,2026-05-11,2026-05-25,None


In [18]:
overdue_books_df = pd.read_sql("""
    SELECT 
        bl.loan_id,
        b.book_title,
        f.first_name,
        f.last_name,
        bl.loan_date,
        bl.due_date,
        DATEDIFF(CURDATE(), bl.due_date) AS days_overdue
    FROM book_loans AS bl
    JOIN books AS b
        ON bl.book_id = b.book_id
    JOIN friends AS f
        ON bl.friend_id = f.friend_id
    WHERE bl.return_date IS NULL
      AND bl.due_date < CURDATE();
""", con=engine)

overdue_books_df

,loan_id,book_title,first_name,last_name,loan_date,due_date,days_overdue
0,32,Norwegian Wood,Liane,Müller,2026-05-11,2026-05-25,19


In [19]:
# Add a new friend from Python to SQL

In [20]:
new_friend_df = pd.DataFrame({
    "first_name": ["Maya"],
    "last_name": ["Keller"],
    "address": ["Teststrasse 1, Essen"],
    "phone_number": ["+49 170 1234567"],
    "max_loan_books": [3],
    "notes": ["Added from Python"]
})

new_friend_df.to_sql(
    "friends",
    con=engine,
    if_exists="append",
    index=False
)

1

In [21]:
import os

os.getcwd()

'/Users/pansupari/Desktop/Projects and Chapters/Chapter 8 Advanced SQL/lianes_library_app'

In [22]:
books_csv_df = pd.read_csv("books_export.csv")

books_csv_df

,book_id,book_title,author_names,genre,availability,book_condition
0,3,Harry Potter and the Philosopher's Stone,J.K. Rowling,Fantasy,0,New
1,5,Murder on the Orient Express,Agatha Christie,Mystery,1,Fair
2,6,The Shining,Stephen King,Horror,0,Fair
3,7,Norwegian Wood,Haruki Murakami,Literary Fiction,0,Good
4,8,Beloved,Toni Morrison,Literary Fiction,0,Very Good
5,9,Sapiens,Yuval Noah Harari,History,1,New
6,10,Atomic Habits,James Clear,Self-help,1,Damaged


In [23]:
new_friend_df = pd.DataFrame({
    "first_name": ["Maya"],
    "last_name": ["Keller"],
    "address": ["Teststrasse 1, Essen"],
    "phone_number": ["+49 170 1234567"],
    "max_loan_books": [3],
    "notes": ["Added from Python"]
})

new_friend_df

,first_name,last_name,address,phone_number,max_loan_books,notes
0,Maya,Keller,"Teststrasse 1, Essen",+49 170 1234567,3,Added from Python


In [24]:
new_friend_df.to_sql(
    "friends",
    con=engine,
    if_exists="append",
    index=False
)

1

In [25]:
pd.read_sql("""
    SELECT *
    FROM friends
    ORDER BY friend_id DESC
    LIMIT 5;
""", con=engine)

,friend_id,first_name,last_name,address,phone_number,max_loan_books,notes
0,18,Maya,Keller,"Teststrasse 1, Essen",+49 170 1234567,3,Added from Python
1,17,Maya,Keller,"Teststrasse 1, Essen",+49 170 1234567,3,Added from Python
2,16,Carsten,Jungkamp,"Ruttenscheider str. 180, Berlin",+49 12386376464,3,Loves cooking books
3,9,Laura,Braun,"Schillerstrasse 30, Wuppertal",+49 178 9999999,2,Likes horror
4,8,Priya,Patel,"Lindenweg 14, Münster",+49 177 8888888,4,Prefers history books


In [26]:
check_friend = pd.read_sql("""
    SELECT *
    FROM friends
    WHERE first_name = 'Maya'
      AND last_name = 'Keller'
      AND phone_number = '+49 170 1234567';
""", con=engine)

if check_friend.empty:
    new_friend_df.to_sql(
        "friends",
        con=engine,
        if_exists="append",
        index=False
    )
    print("Friend inserted.")
else:
    print("Friend already exists. Not inserted.")

Friend already exists. Not inserted.


In [27]:
friends_df

,friend_id,first_name,last_name,address,phone_number,max_loan_books,notes
0,2,Liane,Müller,"Goethestrasse 5, Dortmund",+49 171 2222222,2,Returns books quickly
1,3,Amit,Khanal,"Bahnhofstrasse 18, Bochum",+49 172 3333333,4,Likes fantasy
2,5,Michael,Weber,"Parkallee 7, Düsseldorf",+49 174 5555555,2,Often borrows thrillers
3,6,Anna,Fischer,"Bergstrasse 22, Köln",+49 175 6666666,5,Likes literary fiction
4,7,David,Neumann,"Gartenstrasse 9, Bonn",+49 176 7777777,3,Call before lending
5,8,Priya,Patel,"Lindenweg 14, Münster",+49 177 8888888,4,Prefers history books
6,9,Laura,Braun,"Schillerstrasse 30, Wuppertal",+49 178 9999999,2,Likes horror
7,16,Carsten,Jungkamp,"Ruttenscheider str. 180, Berlin",+49 12386376464,3,Loves cooking books


In [28]:
borrowed_books_df = pd.read_sql("""
    SELECT 
        bl.loan_id,
        b.book_title,
        f.first_name,
        f.last_name,
        bl.loan_date,
        bl.due_date,
        bl.return_date
    FROM book_loans AS bl
    JOIN books AS b
        ON bl.book_id = b.book_id
    JOIN friends AS f
        ON bl.friend_id = f.friend_id
    WHERE bl.return_date IS NULL;
""", con=engine)

borrowed_books_df

,loan_id,book_title,first_name,last_name,loan_date,due_date,return_date
0,29,Harry Potter and the Philosopher's Stone,Liane,Müller,2026-06-01,2026-06-15,None
1,30,Beloved,Michael,Weber,2026-06-01,2026-06-15,None
2,31,The Shining,Anna,Fischer,2026-06-01,2026-06-15,None
3,32,Norwegian Wood,Liane,Müller,2026-05-11,2026-05-25,None
